# Lab — one strategy, one window

The strategy is a file; this notebook only looks at it.

| file | holds |
|---|---|
| `strategies/<name>.py` | **what** is traded — entries, exits, costs, and `COMPARE`, the questions it is asking |
| `lab_config.py` | **which** strategy, **when** (the window) and **how** it is drawn |

Every setting lives in exactly one of them — the lab refuses a setting found in both. A new idea
is a toggle in the strategy plus an arm in its `COMPARE`; a new strategy is a copied file. Whether
anything you see here is real is `evaluate.ipynb`'s job, not this window's.

In [1]:
import sys, os, warnings; warnings.filterwarnings('ignore')
for _p in (os.path.abspath('..'), os.getcwd()):
    if os.path.isfile(os.path.join(_p, 'quant', '__init__.py')) and _p not in sys.path:
        sys.path.insert(0, _p)

from quant.research import fvg_lab, run_arms
from quant.research.workspace import lab_config, list_strategies
from quant.viz import AS_OF_JS, open_in_browser

cfg = lab_config()
print(f"strategy  {cfg.STRATEGY}      (available: {', '.join(list_strategies())})")
print(f"window    {cfg.VIEW_START} -> {cfg.AS_OF or cfg.END}   {cfg.SYMBOL} {cfg.TF}")

strategy  tl_break_confirm      (available: tl_break_confirm)
window    2026-06-01 -> 2026-06-20   BTCUSDT 5m


## 1 · The trades

In [2]:
lab = fvg_lab(cfg)
print(f"{len(lab.trades)} trades  {lab.trades.R.sum():+.2f}R")
lab.trades[["tid", "side", "trigger", "entry_time", "R", "close_reason", "bars_held"]]

[15:56:21] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load
[15:56:21] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[15:56:21] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[15:56:21] INFO | quant.data | cache HIT  binance BTCUSDT 4h -> pushdown load
[15:56:21] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[15:56:21] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[15:56:21] INFO | quant.data | cache HIT  binance BTCUSDT 1m -> pushdown load
[15:56:24] INFO | quant.engine | backtest done | bars=14,401 trades=1 final_cash=9891.45 return=-1.09% dd=3.80% | 44.6 ms
[15:56:24] INFO | quant.engine | backtest done | bars=14,401 trades=7 final_cash=10539.17 return=5.39% dd=5.59% | 0.8 ms


8 trades  +4.02R


,tid,side,trigger,entry_time,R,close_reason,bars_held
0,#01,short,15m,2026-06-01 01:50:00+00:00,2.836125,take_profit,171
1,#02,short,15m,2026-06-02 08:45:00+00:00,2.768037,take_profit,67
2,#03,short,5m,2026-06-05 01:00:00+00:00,2.886505,take_profit,175
3,#04,short,15m,2026-06-09 09:35:00+00:00,-1.173855,stop_loss,599
4,#05,long,5m,2026-06-11 17:45:00+00:00,-1.085464,stop_loss,1994
5,#06,short,5m,2026-06-13 03:25:00+00:00,-1.303342,stop_loss,52
6,#07,short,5m,2026-06-16 02:10:00+00:00,-1.262895,stop_loss,53
7,#08,short,15m,2026-06-17 19:35:00+00:00,0.352932,forced_close_end,629


## 2 · The strategy's questions, on this window

One run per arm of the strategy's `COMPARE`, each changing only what it names. Arms that change
trading settings only re-use everything already loaded, so this costs a second or two per arm.

A window this short decides nothing — it is where every rule in this project has looked good.
Read this table to see *what* an arm does; read `evaluate.ipynb` to see whether it *works*.

In [3]:
arms = run_arms(cfg, base=lab)
arms.table()

[15:56:24] INFO | quant.engine | backtest done | bars=14,401 trades=1 final_cash=9891.45 return=-1.09% dd=3.80% | 0.9 ms
[15:56:24] INFO | quant.engine | backtest done | bars=14,401 trades=7 final_cash=10539.17 return=5.39% dd=5.59% | 0.8 ms
[15:56:24] INFO | quant.engine | backtest done | bars=14,401 trades=2 final_cash=10068.39 return=0.68% dd=2.14% | 0.8 ms
[15:56:24] INFO | quant.engine | backtest done | bars=14,401 trades=7 final_cash=10850.79 return=8.51% dd=2.80% | 0.8 ms
[15:56:24] INFO | quant.engine | backtest done | bars=14,401 trades=1 final_cash=9891.45 return=-1.09% dd=3.80% | 0.9 ms
[15:56:24] INFO | quant.engine | backtest done | bars=14,401 trades=7 final_cash=10870.89 return=8.71% dd=1.96% | 1.5 ms
[15:56:24] INFO | quant.engine | backtest done | bars=14,401 trades=2 final_cash=10068.39 return=0.68% dd=2.14% | 0.9 ms
[15:56:24] INFO | quant.engine | backtest done | bars=14,401 trades=7 final_cash=10870.89 return=8.71% dd=1.96% | 1.5 ms
[15:56:24] INFO | quant.engine |

,run,trades,total R,wins,R / trade,forced_close_end,fvg_exit,stop_loss,take_profit,tp_moved
0,neither,8,4.02,4,0.502,1,0,4,3,0
1,move target,9,8.73,6,0.970,1,0,3,3,2
2,fvg exit,8,7.13,5,0.891,1,2,3,2,0
3,both,9,8.92,6,0.991,1,2,3,2,1
4,no confirmation,13,2.61,5,0.201,1,0,8,4,0


In [4]:
arms.diff()

,side,trigger,neither R,neither exit,move target R,move target exit,fvg exit R,fvg exit exit,both R,both exit,no confirmation R,no confirmation exit
entry,,,,,,,,,,,,
2026-06-01 01:40,short,15m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.80,target
2026-06-01 01:50,short,15m,2.84,target,2.84,target,2.46,fvg exit,2.46,fvg exit,NaN,NaN
2026-06-02 07:00,short,15m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.87,target
2026-06-02 08:45,short,15m,2.77,target,2.77,target,2.77,target,2.77,target,NaN,NaN
2026-06-05 01:00,short,5m,2.89,target,2.89,target,2.89,target,2.89,target,2.89,target
2026-06-05 18:50,short,1h,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.05,stop
2026-06-05 22:50,long,15m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.14,stop
2026-06-06 08:25,long,15m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.14,stop
2026-06-06 21:30,long,15m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.21,stop


## 3 · The setups that did not trade

Every break turned down, with the reason — a filter is only judgeable against what it removed.
On the chart they are one legend entry per reason, all starting off.

In [5]:
lab.rejected.groupby(["tf", "reason"]).size().rename("breaks").reset_index()

,tf,reason,breaks
0,15m,already open,3
1,15m,entry rule,12
2,15m,not confirmed,5
3,1h,not confirmed,1
4,5m,already open,10
5,5m,entry rule,15


## 4 · The chart

In [6]:
fig = lab.figure()
# fig

In [7]:
open_in_browser(fig, f"lab_{cfg.STRATEGY}.html", extra_js=AS_OF_JS if cfg.REWIND else None)

WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/lab_tl_break_confirm.html')

---
### What to change

| want | where |
|---|---|
| look at another strategy | `STRATEGY` in `lab_config.py` |
| another window | `VIEW_START` / `END` in `lab_config.py` |
| change a rule | `strategies/<name>.py` |
| try an idea | a toggle in the strategy, and an arm in its `COMPARE` |
| a new strategy | copy a file in `strategies/`, rename it, point `STRATEGY` at it |
| know whether it is real | `evaluate.ipynb` |